# Tier 2 N=5000 — Re-rodar FT-CUR + SAINT com val_loss

Após a ablação H2 confirmar estatisticamente que `val_loss > val_acc`
para esses modelos (Wilcoxon $p < 10^{-9}$), este experimento revalida
FT-CUR e SAINT em todos os 6 datasets do Tier 2 com o protocolo corrigido.

**Protocolo idêntico ao Tier 2 N=5000** para garantir comparabilidade:
- **Modelos**: FT-CUR (Nyströmformer) + SAINT
- **Datasets**: ADULT, CREDIT, BANK, TELCO, SHOPPERS, HIGGS50K
- **Cap**: N=7143 (→ 5000 train + 2143 test)
- **Tuning**: Optuna TPE, **20 trials × 3-fold CV** (mesmo do Tier 2)
- **Sementes**: 30
- **Total**: 12 tunings + 360 experimentos
- **Diferença vs Tier 2 original**: apenas o `early_stop_metric` (val_loss em vez de val_acc)

**Tempo esperado:**
- T4 GPU: ~6-8h (tuning ~4-5h + experimentos ~2-3h)
- A100: ~3h

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Célula 1: GPU check ─────────────────────────────────────────────────────
!nvidia-smi -L
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')
else:
    print('⚠️ GPU não disponível — Runtime → Change runtime type → GPU')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q numpy scipy scikit-learn pandas optuna xgboost xlrd pyarrow

import numpy, scipy, sklearn, torch
print(f'numpy {numpy.__version__} | torch {torch.__version__}')

In [ ]:
# ── Célula 4: Baixar todos os 6 datasets do Tier 2 ─────────────────────────
!python scripts/download_data.py --tier 2
!ls -lh data/raw/ | head -15

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/tuning', exist_ok=True)
print(f'Drive: {DRIVE_PATH}')

In [ ]:
# ── Célula 6: Restaurar progresso anterior (resume) ─────────────────────────
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

# Resultados de experimentos (val_loss)
src = drive_results / 'tier2_n5000_ftcur_saint_valloss.json'
if src.exists():
    shutil.copy(src, 'results/tier2_n5000_ftcur_saint_valloss.json')
    print('✓ Restaurado: tier2_n5000_ftcur_saint_valloss.json')
else:
    print('• Começando do zero (experimentos)')

# Params tunados com val_loss
src_p = drive_results / 'tuning' / 'best_params_ftcur_saint_valloss.json'
if src_p.exists():
    shutil.copy(src_p, 'results/tuning/best_params_ftcur_saint_valloss.json')
    import json
    n = len(json.load(open('results/tuning/best_params_ftcur_saint_valloss.json')))
    print(f'✓ Restaurado: {n} combos tunados com val_loss')
else:
    print('• Tuning vai começar do zero')

In [ ]:
# ── Célula 7: Sync para Drive a cada 5 min (em background) ──────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier2_n5000_ftcur_saint_valloss.json \
          "$1/tier2_n5000_ftcur_saint_valloss.json" 2>/dev/null
    mkdir -p "$1/tuning"
    cp -u /content/sparse-lssvm-transformers-study/results/tuning/best_params_ftcur_saint_valloss.json \
          "$1/tuning/best_params_ftcur_saint_valloss.json" 2>/dev/null
done


In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid}) — salva a cada 5 min')


In [ ]:
# ── Célula 8: Rodar tuning + experimentos ───────────────────────────────────
# IDÊNTICO AO TIER 2 N=5000 PARA COMPARABILIDADE DIRETA:
#   Tuning: 2 modelos × 6 datasets × 20 trials × 3-fold CV = 12 combos
#   Experimentos: 2 × 6 × 30 = 360 runs
#   Subsample cap: N=7143 (mesmo do Tier 2)
#
# Em T4: ~6-8h total (tuning ~4-5h + experimentos ~2-3h)
# Em A100: ~3h
#
# OBS: 20 trials (não 15) é obrigatório para que os resultados sejam
# substitutos válidos dos FT-CUR/SAINT no Tier 2 imbalanceado.

!python scripts/run_ftcur_saint_rerun.py \
    --early-stop-metric val_loss \
    --datasets ADULT CREDIT BANK TELCO SHOPPERS HIGGS50K \
    --output-file results/tier2_n5000_ftcur_saint_valloss.json \
    --params-file results/tuning/best_params_ftcur_saint_valloss.json \
    --seeds 30 \
    --trials 20 \
    --folds 3

In [ ]:
# ── Célula 9: Salvar resultado final no Drive ───────────────────────────────
import shutil, signal
try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

from pathlib import Path
import os
drive_dest = Path(DRIVE_PATH)
(drive_dest / 'tuning').mkdir(exist_ok=True)

for fname in ['tier2_n5000_ftcur_saint_valloss.json',
              'tuning/best_params_ftcur_saint_valloss.json']:
    src = Path('results') / fname
    dst = drive_dest / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive ({DRIVE_PATH}):')
!ls -lh '{DRIVE_PATH}'